# Autoencoder for Image Denoising using MNIST

**Assignment:** Build a deep learning model that can remove noise from images using an autoencoder on MNIST.

This notebook implements a **Convolutional Denoising Autoencoder**. Gaussian noise is added to clean MNIST images. The network receives noisy images as input and learns to reconstruct the corresponding clean images.

### Objectives
- Load and preprocess MNIST handwritten digit images
- Add artificial Gaussian noise
- Build a convolutional encoder-decoder network
- Train the model using noisy → clean image pairs
- Evaluate reconstruction quality using MSE and PSNR
- Visualize noisy, denoised, and original images
- Save the trained model


In [ ]:
# 1. Import libraries
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## Dataset

MNIST contains grayscale handwritten digit images of size **28 × 28**. `ToTensor()` converts each image to a tensor with pixel values in the range `[0, 1]`.


In [ ]:
# 2. Load MNIST
transform = transforms.ToTensor()

full_train = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_dataset, val_dataset = random_split(
    full_train, [54000, 6000],
    generator=torch.Generator().manual_seed(SEED)
)

BATCH_SIZE = 128
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))


In [ ]:
# 3. Function to add Gaussian noise
NOISE_FACTOR = 0.40

def add_noise(images, noise_factor=NOISE_FACTOR):
    noisy = images + noise_factor * torch.randn_like(images)
    return torch.clamp(noisy, 0.0, 1.0)

# Visualize original and noisy samples
images, _ = next(iter(train_loader))
noisy_images = add_noise(images)

fig, axes = plt.subplots(2, 8, figsize=(12, 4))
for i in range(8):
    axes[0, i].imshow(images[i].squeeze(), cmap="gray")
    axes[0, i].axis("off")
    axes[1, i].imshow(noisy_images[i].squeeze(), cmap="gray")
    axes[1, i].axis("off")
axes[0, 0].set_ylabel("Original")
axes[1, 0].set_ylabel("Noisy")
plt.suptitle("Original vs Noisy MNIST Images")
plt.tight_layout()
plt.show()


## Model Architecture

The **encoder** uses convolutional layers to extract important features and compress the image. The **decoder** uses transposed convolutions to reconstruct a clean 28×28 image. A sigmoid output keeps reconstructed pixels between 0 and 1.


In [ ]:
# 4. Convolutional Denoising Autoencoder
class DenoisingAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1),  # 28 -> 14
            nn.ReLU(inplace=True),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1), # 14 -> 7
            nn.ReLU(inplace=True)
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1), # 7 -> 14
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(16, 1, kernel_size=4, stride=2, padding=1),  # 14 -> 28
            nn.Sigmoid()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

model = DenoisingAutoencoder().to(device)
print(model)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")


In [ ]:
# 5. Loss function and optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 10


In [ ]:
# 6. Train the autoencoder
train_losses = []
val_losses = []

for epoch in range(EPOCHS):
    model.train()
    running_train_loss = 0.0

    for clean_images, _ in train_loader:
        clean_images = clean_images.to(device)
        noisy_images = add_noise(clean_images)

        optimizer.zero_grad()
        outputs = model(noisy_images)
        loss = criterion(outputs, clean_images)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * clean_images.size(0)

    train_loss = running_train_loss / len(train_loader.dataset)
    train_losses.append(train_loss)

    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for clean_images, _ in val_loader:
            clean_images = clean_images.to(device)
            noisy_images = add_noise(clean_images)
            outputs = model(noisy_images)
            loss = criterion(outputs, clean_images)
            running_val_loss += loss.item() * clean_images.size(0)

    val_loss = running_val_loss / len(val_loader.dataset)
    val_losses.append(val_loss)

    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")


In [ ]:
# 7. Plot training and validation loss
plt.figure(figsize=(8, 5))
plt.plot(range(1, EPOCHS + 1), train_losses, marker="o", label="Training Loss")
plt.plot(range(1, EPOCHS + 1), val_losses, marker="o", label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Autoencoder Training History")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
# 8. Evaluate on unseen test images
model.eval()
test_loss = 0.0

with torch.no_grad():
    for clean_images, _ in test_loader:
        clean_images = clean_images.to(device)
        noisy_images = add_noise(clean_images)
        outputs = model(noisy_images)
        test_loss += criterion(outputs, clean_images).item() * clean_images.size(0)

test_mse = test_loss / len(test_loader.dataset)
psnr = 10 * np.log10(1.0 / test_mse)

print(f"Test MSE : {test_mse:.6f}")
print(f"Test PSNR: {psnr:.2f} dB")


In [ ]:
# 9. Compare Noisy -> Denoised -> Original
clean_images, _ = next(iter(test_loader))
clean_images = clean_images.to(device)
noisy_images = add_noise(clean_images)

with torch.no_grad():
    denoised_images = model(noisy_images)

clean_images = clean_images.cpu()
noisy_images = noisy_images.cpu()
denoised_images = denoised_images.cpu()

n = 10
fig, axes = plt.subplots(3, n, figsize=(15, 5))

for i in range(n):
    axes[0, i].imshow(noisy_images[i].squeeze(), cmap="gray")
    axes[0, i].axis("off")
    axes[1, i].imshow(denoised_images[i].squeeze(), cmap="gray")
    axes[1, i].axis("off")
    axes[2, i].imshow(clean_images[i].squeeze(), cmap="gray")
    axes[2, i].axis("off")

axes[0, 0].set_ylabel("Noisy")
axes[1, 0].set_ylabel("Denoised")
axes[2, 0].set_ylabel("Original")
plt.suptitle("MNIST Image Denoising Results")
plt.tight_layout()
plt.show()


In [ ]:
# 10. Save the trained model
os.makedirs("model", exist_ok=True)
torch.save(model.state_dict(), "model/mnist_denoising_autoencoder.pth")
print("Model saved to model/mnist_denoising_autoencoder.pth")


## Conclusion

A convolutional denoising autoencoder was implemented for MNIST handwritten digits. Artificial Gaussian noise was added to clean images and the network was trained with **noisy images as inputs** and **clean images as targets**. The encoder learns a compact feature representation while the decoder reconstructs the image. MSE measures pixel-level reconstruction error, while PSNR provides an additional measure of reconstruction quality.

### Key Learning Outcomes
- Understanding encoder-decoder architecture
- Image preprocessing and artificial noise generation
- Convolution and transposed-convolution layers
- Training a deep neural network with PyTorch
- MSE-based reconstruction learning
- Evaluating and visualizing image-denoising results
